# 05b ML Fixed Embeddings

This notebook mirrors `05_ML_GNN_Embeddings.ipynb` but uses the fixed embedding datasets produced by `03b_fixembeddings.ipynb`.

| Dataset | Dim | What changed |
|---|---|---|
| `graphsage_fixed_srisk_dataset.parquet` | 32 | Reconstruction loss instead of link prediction; 32-dim bottleneck |
| `node2vec_fixed_srisk_dataset.parquet` | 32 | 32-dim purely structural (was 64); warm-starting for temporal coherence |

Both models are **graph-only** — no financial features added — keeping the comparison against classical centrality measures fair.

> **Note:** run `03b_fixembeddings.ipynb` first to generate the parquet files.

In [1]:
from pathlib import Path
import sys
import os

sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

from src.models.ml_train_and_store import (
    ModelTrainer,
    load_gnn_dataset,
    make_pipeline,
)

pd.set_option('display.max_columns', 200)
PROJECT_ROOT = Path().resolve().parents[1]
print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis


## Load Datasets

In [2]:
DISPLAY_COLS = ["model", "train_mae", "validation_mae", "train_rmse", "validation_rmse"]

# Fixed GraphSAGE: 32-dim reconstruction-trained embeddings
df_sage, feature_cols_sage = load_gnn_dataset(
    PROJECT_ROOT,
    target_col="log_systemic_risk_label",
    filename="graphsage_fixed_srisk_dataset.parquet",
)
print(f"GraphSAGE fixed:    {df_sage.shape}  —  {len(feature_cols_sage)} embedding cols (32-dim)")

# Node2Vec v2: 32-dim purely structural embeddings
df_n2v, feature_cols_n2v = load_gnn_dataset(
    PROJECT_ROOT,
    target_col="log_systemic_risk_label",
    filename="node2vec_fixed_srisk_dataset.parquet",
)
print(f"Node2Vec v2 fixed:  {df_n2v.shape}  —  {len(feature_cols_n2v)} embedding cols (32-dim)")

# Node2Vec v3: 128-dim structural embeddings
df_n2v_v3, feature_cols_n2v_v3 = load_gnn_dataset(
    PROJECT_ROOT,
    target_col="log_systemic_risk_label",
    filename="node2vec_fixed_v3_srisk_dataset.parquet",
)
print(f"Node2Vec v3 fixed:  {df_n2v_v3.shape}  —  {len(feature_cols_n2v_v3)} embedding cols (128-dim)")

GraphSAGE fixed:    (145536, 37)  —  32 embedding cols (32-dim)
Node2Vec v2 fixed:  (145536, 37)  —  32 embedding cols (32-dim)
Node2Vec v3 fixed:  (145536, 133)  —  128 embedding cols (128-dim)


In [3]:
trainer_sage = ModelTrainer(df=df_sage, feature_cols=feature_cols_sage, target_col="log_systemic_risk_label")
trainer_n2v  = ModelTrainer(df=df_n2v,  feature_cols=feature_cols_n2v,  target_col="log_systemic_risk_label")
trainer_n2v_v3 = ModelTrainer(df=df_n2v_v3, feature_cols=feature_cols_n2v_v3, target_col="log_systemic_risk_label")

print("GraphSAGE fixed   —", trainer_sage.train_df.shape, trainer_sage.val_df.shape)
print("Node2Vec v2 fixed —", trainer_n2v.train_df.shape,  trainer_n2v.val_df.shape)
print("Node2Vec v3 fixed —", trainer_n2v_v3.train_df.shape, trainer_n2v_v3.val_df.shape)

GraphSAGE fixed   — (109152, 37) (18192, 37)
Node2Vec v2 fixed — (109152, 37) (18192, 37)
Node2Vec v3 fixed — (109152, 133) (18192, 133)


## Define Models

In [4]:
candidate_models = {
    "linear_regression": make_pipeline(LinearRegression(), scale_features=True),
    "ridge":             make_pipeline(Ridge(alpha=1.0), scale_features=True),
    "random_forest":     make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42)),
    "xgboost":           make_pipeline(XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42), scale_features=True),
}

list(candidate_models)

['linear_regression', 'ridge', 'random_forest', 'xgboost']

## Train — Fixed GraphSAGE

In [5]:
trainer_sage.train_all(candidate_models)
trainer_sage.leaderboard()[DISPLAY_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,random_forest,0.006006,0.129821,0.033576,0.354567
1,xgboost,0.011602,0.162925,0.05398,0.397317
2,linear_regression,0.050143,0.18192,0.154467,0.50086
3,ridge,0.050196,0.309518,0.153664,0.839892


In [6]:
trainer_sage.test_predictions().head(20)

,bank_id,year,quarter,period,log_systemic_risk_label,prediction,abs_error
0,796,2023,1,2023Q1,0.693147,2.642224,1.949077
1,1889,2023,1,2023Q1,0.693147,2.628372,1.935225
2,2474,2023,1,2023Q1,0.693147,2.626549,1.933402
3,1957,2023,1,2023Q1,0.693147,2.626549,1.933402
4,2958,2023,1,2023Q1,0.693147,2.626549,1.933402
5,2501,2023,1,2023Q1,0.693147,2.626549,1.933402
6,3406,2023,1,2023Q1,0.693147,2.626549,1.933402
7,3218,2023,1,2023Q1,0.693147,2.626549,1.933402
8,266,2023,1,2023Q1,0.693147,2.626549,1.933402
9,4189,2023,1,2023Q1,0.693147,2.618076,1.924929


## Train — Enriched Node2Vec

In [7]:
trainer_n2v.train_all(candidate_models)
trainer_n2v.leaderboard()[DISPLAY_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,random_forest,0.006772,0.032153,0.032031,0.131996
1,xgboost,0.013355,0.030482,0.062539,0.136432
2,linear_regression,0.058798,0.080441,0.144037,0.197041
3,ridge,0.058785,0.080419,0.144037,0.197046


## Train — Node2Vec v3 (128-dim)

In [8]:
trainer_n2v_v3.train_all(candidate_models)
trainer_n2v_v3.leaderboard()[DISPLAY_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,random_forest,0.006494,0.030423,0.031862,0.129201
1,xgboost,0.011188,0.029773,0.050109,0.130405
2,ridge,0.055,0.077092,0.135624,0.212803
3,linear_regression,0.055009,0.077104,0.135624,0.212804


In [9]:
trainer_n2v.test_predictions().head(20)

,bank_id,year,quarter,period,log_systemic_risk_label,prediction,abs_error
0,2,2023,1,2023Q1,3.713572,2.065521,1.648051
1,8,2023,3,2023Q3,3.637586,2.064287,1.573299
2,53,2023,2,2023Q2,2.397895,0.877706,1.520189
3,878,2023,1,2023Q1,0.693147,2.211546,1.518399
4,17,2023,2,2023Q2,3.637586,2.150226,1.487360
5,6,2023,2,2023Q2,3.610918,2.129710,1.481207
6,4,2023,2,2023Q2,3.555348,2.076551,1.478797
7,926,2023,1,2023Q1,0.693147,2.136369,1.443222
8,3,2023,1,2023Q1,2.772589,1.367747,1.404842
9,2,2023,2,2023Q2,3.465736,2.089362,1.376374


## Best Models

In [10]:
print("GraphSAGE fixed best:  ", trainer_sage.best_name())
print("Node2Vec enriched best:", trainer_n2v.best_name())

GraphSAGE fixed best:   random_forest
Node2Vec enriched best: random_forest


## Comparison with Original Embeddings

Reference results from `04_ML_Classic_Algorithms.ipynb` and `05_ML_GNN_Embeddings.ipynb` (best model per dataset, no warm-starting):

| Dataset | Best model | Train MAE | Val MAE | Train RMSE | Val RMSE |
|---|---|---|---|---|---|
| Classical features (DebtRank, PageRank, ...) | XGBRegressor | 0.0360 | 0.1253 | 0.4301 | 1.6875 |
| GraphSAGE original (link prediction, 64-dim) | MLP | 0.0860 | 0.2428 | 1.3481 | 3.0198 |
| Node2Vec original (64-dim) | — | — | — | — | — |

> Node2Vec original never ran — missing `torch-cluster` dependency.

Fixed embeddings (this notebook) — best model per dataset:

In [11]:
def best_row(trainer, label):
    row = trainer.leaderboard().iloc[0]
    return {
        "Dataset": label,
        "Best model": row["model"],
        "Train MAE": round(float(row["train_mae"]), 4),
        "Val MAE":   round(float(row["validation_mae"]), 4),
        "Train RMSE": round(float(row["train_rmse"]), 4),
        "Val RMSE":   round(float(row["validation_rmse"]), 4),
    }

comparison = pd.DataFrame([
    best_row(trainer_sage,    "GraphSAGE fixed (reconstruction, 32-dim)"),
    best_row(trainer_n2v,     "Node2Vec v2 fixed (structural, 32-dim)"),
    best_row(trainer_n2v_v3,  "Node2Vec v3 fixed (structural, 128-dim)"),
])

comparison.set_index("Dataset")

,Best model,Train MAE,Val MAE,Train RMSE,Val RMSE
Dataset,,,,,
"GraphSAGE fixed (reconstruction, 32-dim)",random_forest,0.0060,0.1298,0.0336,0.3546
"Node2Vec v2 fixed (structural, 32-dim)",random_forest,0.0068,0.0322,0.0320,0.1320
"Node2Vec v3 fixed (structural, 128-dim)",random_forest,0.0065,0.0304,0.0319,0.1292
